# Fleet Data Skew Reprocessing

Rewrites the Aircraft Predictive Maintenance source tables in `genie_zeroops_mfg_catalog.aircraft_maintenance` with realistic skew so the dashboard tells a compelling story. Backs up all tables first.

In [0]:
%sql
-- Back up all 4 source tables before any changes
CREATE TABLE IF NOT EXISTS genie_zeroops_mfg_catalog.aircraft_maintenance.flights_metadata_backup_20261021
  AS SELECT * FROM genie_zeroops_mfg_catalog.aircraft_maintenance.flights_metadata;

CREATE TABLE IF NOT EXISTS genie_zeroops_mfg_catalog.aircraft_maintenance.raw_flight_telemetry_backup_20261021
  DEEP CLONE genie_zeroops_mfg_catalog.aircraft_maintenance.raw_flight_telemetry;

CREATE TABLE IF NOT EXISTS genie_zeroops_mfg_catalog.aircraft_maintenance.aircraft_registry_backup_20261021
  AS SELECT * FROM genie_zeroops_mfg_catalog.aircraft_maintenance.aircraft_registry;

CREATE TABLE IF NOT EXISTS genie_zeroops_mfg_catalog.aircraft_maintenance.staging_flight_telemetry_api_backup_20261021
  AS SELECT * FROM genie_zeroops_mfg_catalog.aircraft_maintenance.staging_flight_telemetry_api;

SELECT 'Backups complete' AS status,
  (SELECT COUNT(*) FROM genie_zeroops_mfg_catalog.aircraft_maintenance.flights_metadata_backup_20261021) AS flights_bak,
  (SELECT COUNT(*) FROM genie_zeroops_mfg_catalog.aircraft_maintenance.raw_flight_telemetry_backup_20261021) AS telemetry_bak,
  (SELECT COUNT(*) FROM genie_zeroops_mfg_catalog.aircraft_maintenance.aircraft_registry_backup_20261021) AS registry_bak

In [0]:
import random
import builtins
import uuid
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

random.seed(42)

SRC = "genie_zeroops_mfg_catalog.aircraft_maintenance"
DST = SRC  # overwriting in place

# ── Target distribution ──
AIRLINE_PROFILE = {
    # airline: (target_flights, anomaly_rate, severity_critical_high_pct, primary_anomaly_types, on_time_base)
    'SkyWest Global':        (220, 0.22, 0.35, ['VIBRATION_ANOMALY','ENGINE_DEGRADATION','COMPRESSOR_STALL'], 0.78),
    'Atlas Pacific':         (180, 0.20, 0.30, ['BLEED_AIR_LEAK','COMPRESSOR_STALL','OIL_SYSTEM_FAULT'], 0.82),
    'Continental Wings':     (110, 0.14, 0.20, ['BRAKE_OVERHEAT','HYDRAULIC_LEAK'], 0.87),
    'Horizon Express':       (120, 0.12, 0.18, ['FUEL_IMBALANCE','ELECTRICAL_FAULT'], 0.89),
    'Falcon Jet Airways':    (90,  0.11, 0.16, ['COMPRESSOR_STALL','SENSOR_DRIFT'], 0.90),
    'TransOcean Air':        (250, 0.08, 0.12, ['BLEED_AIR_LEAK','OIL_SYSTEM_FAULT'], 0.93),
    'Meridian Airlines':     (130, 0.09, 0.14, ['HYDRAULIC_LEAK','ELECTRICAL_FAULT'], 0.92),
    'Northern Star Airways': (150, 0.06, 0.10, ['VIBRATION_ANOMALY','SENSOR_DRIFT'], 0.95),
    'Aurora Airlines':       (70,  0.04, 0.08, ['OIL_SYSTEM_FAULT'], 0.97),
    'Summit Aviation':       (60,  0.03, 0.06, ['ELECTRICAL_FAULT'], 0.98),
}

ALL_ANOMALY_TYPES = [
    'VIBRATION_ANOMALY','ENGINE_DEGRADATION','BLEED_AIR_LEAK','COMPRESSOR_STALL',
    'OIL_SYSTEM_FAULT','BRAKE_OVERHEAT','HYDRAULIC_LEAK','ELECTRICAL_FAULT',
    'SENSOR_DRIFT','FUEL_IMBALANCE'
]
SEVERITIES = ['CRITICAL','HIGH','MEDIUM','LOW']

ORIGINS = ['JFK','BOS','MIA','SYD','HND','FRA','GRU','SIN','DEN','SEA',
           'ORD','DXB','LAX','CDG','LHR','YYZ','ATL','AMS','MAD','SFO']
DESTS = ORIGINS

# Load existing tail numbers per airline from registry
registry_df = spark.read.table(f"{SRC}.aircraft_registry")
airline_tails = {}
for row in registry_df.select('airline','tail_number','aircraft_type').collect():
    airline_tails.setdefault(row.airline, []).append((row.tail_number, row.aircraft_type))

# Generate monthly trend: problem carriers get worse Jul-Oct 2026
TREND = {
    'SkyWest Global': {'base': 0.18, (2026, 7): 0.24, (2026, 8): 0.28, (2026, 9): 0.33, (2026, 10): 0.38},
    'Atlas Pacific':  {'base': 0.16, (2026, 7): 0.20, (2026, 8): 0.24, (2026, 9): 0.28, (2026, 10): 0.32},
}

def monthly_rate(airline, month_key, base_rate):
    """Problem carriers follow TREND per (year, month); everyone else uses the flat base rate."""
    if month_key is None or airline not in TREND:
        return base_rate
    return TREND[airline].get(month_key, TREND[airline]['base'])

def pick_severity(crit_high_pct):
    r = random.random()
    if r < crit_high_pct * 0.4:
        return 'CRITICAL'
    elif r < crit_high_pct:
        return 'HIGH'
    elif r < crit_high_pct + (1 - crit_high_pct) * 0.6:
        return 'MEDIUM'
    else:
        return 'LOW'

def pick_anomaly_type(primary_types):
    if random.random() < 0.7:
        return random.choice(primary_types)
    else:
        return random.choice(ALL_ANOMALY_TYPES)

# Generate flights
flight_counter = 0
rows = []
start_date = datetime(2025, 9, 1)
end_date = datetime(2026, 10, 15)
date_range_seconds = int((end_date - start_date).total_seconds())

for airline, (target_flights, base_anom_rate, crit_high_pct, primary_types, _) in AIRLINE_PROFILE.items():
    tails = airline_tails.get(airline, [('N99999', 'A320neo')])
    airline_flights = []
    
    # Distribute flights across months with slight increase over time
    for i in range(target_flights):
        flight_counter += 1
        flight_id = f"FLT_{flight_counter:05d}"
        
        # Spread departure times across the date range
        dep_offset = random.randint(0, date_range_seconds)
        dep_time = start_date + timedelta(seconds=dep_offset)
        duration_hrs = builtins.round(random.uniform(1.5, 6.5), 2)
        arr_time = dep_time + timedelta(hours=duration_hrs)
        
        tail_num, ac_type = random.choice(tails)
        origin = random.choice(ORIGINS)
        dest = random.choice([d for d in DESTS if d != origin])
        
        airline_flights.append(dict(
            flight_id=flight_id, airline=airline, aircraft_type=ac_type, tail_number=tail_num,
            origin=origin, destination=dest, departure_time=dep_time, arrival_time=arr_time,
        ))

    # Deterministic anomaly assignment so each carrier lands on its target rate.
    # Problem carriers: exact count per month (rising Jul-Oct 2026). Others: exact count per airline.
    if airline in TREND:
        groups = {}
        for f in airline_flights:
            groups.setdefault((f['departure_time'].year, f['departure_time'].month), []).append(f)
        groups = list(groups.items())
    else:
        groups = [(None, airline_flights)]

    for month_key, fl in groups:
        rate = monthly_rate(airline, month_key, base_anom_rate)
        k = int(builtins.round(len(fl) * rate))
        chosen = set(random.sample(range(len(fl)), k))
        for idx, f in enumerate(fl):
            is_anom = idx in chosen
            rows.append(Row(
                flight_id=f['flight_id'], airline=f['airline'], aircraft_type=f['aircraft_type'],
                tail_number=f['tail_number'], origin=f['origin'], destination=f['destination'],
                departure_time=f['departure_time'], arrival_time=f['arrival_time'],
                has_anomaly=is_anom,
                anomaly_type=pick_anomaly_type(primary_types) if is_anom else None,
                anomaly_severity=pick_severity(crit_high_pct) if is_anom else None,
            ))

schema = StructType([
    StructField('flight_id', StringType()),
    StructField('airline', StringType()),
    StructField('aircraft_type', StringType()),
    StructField('tail_number', StringType()),
    StructField('origin', StringType()),
    StructField('destination', StringType()),
    StructField('departure_time', TimestampType()),
    StructField('arrival_time', TimestampType()),
    StructField('has_anomaly', BooleanType()),
    StructField('anomaly_type', StringType()),
    StructField('anomaly_severity', StringType()),
])

new_flights_df = spark.createDataFrame(rows, schema)

# Summary
print(f"Total flights generated: {new_flights_df.count()}")
display(
    new_flights_df.groupBy('airline').agg(
        F.count('*').alias('flights'),
        F.sum(F.when(F.col('has_anomaly'), 1).otherwise(0)).alias('anomalies'),
        F.round(F.avg(F.when(F.col('has_anomaly'), 1.0).otherwise(0.0)) * 100, 1).alias('anom_pct')
    ).orderBy(F.desc('anom_pct'))
)

In [0]:
# Overwrite the source flights_metadata table
new_flights_df.write.mode('overwrite').saveAsTable(f"{SRC}.flights_metadata")
print(f"flights_metadata overwritten: {spark.read.table(f'{SRC}.flights_metadata').count()} rows")

In [0]:
# Strategy: for each new flight, clone telemetry from a random existing backup flight,
# remapping flight_id, airline, aircraft_type, tail_number, origin, destination,
# timestamps, and anomaly columns.

from pyspark.sql.window import Window

# Read backup telemetry and new flights
old_telem = spark.read.table(f"{SRC}.raw_flight_telemetry_backup_20261021")
new_flights = spark.read.table(f"{SRC}.flights_metadata")

# Get list of old flight IDs available as templates
old_flight_ids = [r.flight_id for r in old_telem.select('flight_id').distinct().collect()]
print(f"Template flights available: {len(old_flight_ids)}")

# Create mapping: new_flight_id -> template flight, plus an anomaly window for anomalous flights.
# Anomalies are flagged only on a contiguous slice of one phase (CLIMB ~70%, CRUISE ~30%) so the
# pipeline's peak phase and event duration stay realistic for the Flight Phase Analysis tab.
import random
random.seed(42)
PHASE_LEN = {'CLIMB': 150, 'CRUISE': 549}
mapping_rows = []
for row in new_flights.collect():
    template_fid = random.choice(old_flight_ids)
    if row.has_anomaly:
        anom_phase = 'CLIMB' if random.random() < 0.70 else 'CRUISE'
        win_len = random.randint(30, 110) if anom_phase == 'CLIMB' else random.randint(60, 300)
        win_start = random.randint(1, PHASE_LEN[anom_phase] - win_len)
    else:
        anom_phase, win_len, win_start = '', 0, 0
    mapping_rows.append(Row(
        new_flight_id=row.flight_id,
        template_flight_id=template_fid,
        new_airline=row.airline,
        new_aircraft_type=row.aircraft_type,
        new_tail_number=row.tail_number,
        new_origin=row.origin,
        new_destination=row.destination,
        new_departure_time=row.departure_time,
        new_arrival_time=row.arrival_time,
        new_anomaly_type=row.anomaly_type,
        new_anomaly_severity=row.anomaly_severity,
        anom_phase=anom_phase,
        win_start=win_start,
        win_len=win_len,
    ))

mapping_df = spark.createDataFrame(mapping_rows)
print(f"Flight mappings: {mapping_df.count()}")

# Join template telemetry with mapping and remap columns
# Position of each record within its flight phase (used to place the anomaly window)
old_telem_idx = old_telem.withColumn(
    '_phase_idx',
    F.row_number().over(Window.partitionBy('flight_id', 'flight_phase').orderBy('record_sequence'))
)

new_telem = (
    old_telem_idx.join(
        mapping_df,
        old_telem_idx.flight_id == mapping_df.template_flight_id,
        'inner'
    )
    # Replace metadata columns with new flight's values
    .drop('flight_id', 'airline', 'aircraft_type', 'tail_number',
           'origin', 'destination', 'anomaly_type', 'anomaly_severity')
    .withColumnRenamed('new_flight_id', 'flight_id')
    .withColumnRenamed('new_airline', 'airline')
    .withColumnRenamed('new_aircraft_type', 'aircraft_type')
    .withColumnRenamed('new_tail_number', 'tail_number')
    .withColumnRenamed('new_origin', 'origin')
    .withColumnRenamed('new_destination', 'destination')
    .withColumnRenamed('new_anomaly_type', '_new_anom_type')
    .withColumnRenamed('new_anomaly_severity', '_new_anom_sev')
    .drop('template_flight_id', 'new_departure_time', 'new_arrival_time')
)

# The pipeline expects anomaly_type/severity on every record of an anomalous flight (one row per
# flight in gold_flight_summary), and picks flight_phase_at_peak from the max-EGT record.
# So keep the labels flight-wide and inject the sensor signature only inside the window.
in_win = (F.col('flight_phase') == F.col('anom_phase')) & F.col('_phase_idx').between(
    F.col('win_start'), F.col('win_start') + F.col('win_len'))
new_telem = new_telem.withColumn('anomaly_type', F.col('_new_anom_type')) \
    .withColumn('anomaly_severity', F.col('_new_anom_sev'))
# CRUISE runs ~50C cooler than CLIMB, so it needs a larger bump for its window to hold the peak
egt_bump = F.when(F.col('anom_phase') == 'CRUISE', F.lit(130)).otherwise(F.lit(45))
for c in ['egt_eng1', 'egt_eng2']:
    new_telem = new_telem.withColumn(c, F.when(in_win, F.col(c) + egt_bump).otherwise(F.col(c)))
for c in ['vibration_n1_eng1', 'vibration_n2_eng1', 'vibration_n1_eng2', 'vibration_n2_eng2']:
    new_telem = new_telem.withColumn(c, F.when(in_win, F.col(c) * 1.8).otherwise(F.col(c)))
new_telem = new_telem.drop('_new_anom_type', '_new_anom_sev', 'anom_phase', 'win_start', 'win_len', '_phase_idx')

# Retime timestamps: shift each flight's telemetry to match new departure_time
# First get the min timestamp per template flight to compute offsets
w = Window.partitionBy('flight_id')
new_telem = new_telem.withColumn('_min_ts', F.min('timestamp').over(w))

# Join with new flights to get departure_time for offset
new_telem = new_telem.join(
    new_flights.select(
        F.col('flight_id').alias('_fid'),
        F.col('departure_time').alias('_new_dep')
    ),
    new_telem.flight_id == F.col('_fid'),
    'left'
).drop('_fid')

# Compute new timestamps
new_telem = (
    new_telem
    .withColumn('timestamp',
        F.from_unixtime(
            F.unix_timestamp('_new_dep') + 
            (F.unix_timestamp('timestamp') - F.unix_timestamp('_min_ts'))
        ).cast('timestamp')
    )
    .withColumn('event_date', F.to_date('timestamp'))
    .drop('_min_ts', '_new_dep')
)

print(f"New telemetry rows: {new_telem.count()}")
print(f"Distinct flights in telemetry: {new_telem.select('flight_id').distinct().count()}")

In [0]:
# Reorder columns to match original schema
original_cols = spark.read.table(f"{SRC}.raw_flight_telemetry_backup_20261021").columns
new_telem_ordered = new_telem.select(*original_cols)

# Overwrite telemetry
new_telem_ordered.write.mode('overwrite').saveAsTable(f"{SRC}.raw_flight_telemetry")
print(f"raw_flight_telemetry overwritten: {spark.read.table(f'{SRC}.raw_flight_telemetry').count()} rows")

# Update aircraft_registry: make problem carrier fleets older
registry = spark.read.table(f"{SRC}.aircraft_registry_backup_20261021")

age_adjustments = {
    'SkyWest Global': 4,      # +4 years → avg ~14 yrs
    'Atlas Pacific': 6,       # +6 years → avg ~18 yrs (oldest fleet)
    'Continental Wings': 2,
    'Aurora Airlines': -3,    # newer fleet
    'Summit Aviation': -4,    # newest fleet
}

from pyspark.sql.functions import when, col, lit, add_months, expr

for airline, adj in age_adjustments.items():
    registry = registry.withColumn(
        'aircraft_age_years',
        when(col('airline') == airline, col('aircraft_age_years') + lit(adj))
        .otherwise(col('aircraft_age_years'))
    ).withColumn(
        'manufacture_date',
        when(col('airline') == airline, add_months(col('manufacture_date'), -adj * 12))
        .otherwise(col('manufacture_date'))
    )

# Ensure no negative ages
registry = registry.withColumn(
    'aircraft_age_years',
    when(col('aircraft_age_years') < 1, lit(1)).otherwise(col('aircraft_age_years'))
)

registry.write.mode('overwrite').saveAsTable(f"{SRC}.aircraft_registry")
print(f"aircraft_registry overwritten: {spark.read.table(f'{SRC}.aircraft_registry').count()} rows")

# Show age profile
display(
    spark.read.table(f"{SRC}.aircraft_registry")
    .groupBy('airline')
    .agg(F.round(F.avg('aircraft_age_years'), 1).alias('avg_age_yrs'), F.count('*').alias('fleet_size'))
    .orderBy(F.desc('avg_age_yrs'))
)

In [0]:
# Trigger a full refresh of the Aircraft Maintenance Medallion Pipeline
# This rebuilds all streaming tables and MVs from the updated source data
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

PIPELINE_ID = "5bd5dac4-6cae-4af6-960d-056a266e7143"

print("Starting full pipeline refresh...")
update = w.pipelines.start_update(
    pipeline_id=PIPELINE_ID,
    full_refresh=True  # forces recompute of all streaming tables and MVs
)
print(f"Pipeline update started: {update.update_id}")
print(f"Monitor at: {w.config.host}/#joblist/pipelines/{PIPELINE_ID}/updates/{update.update_id}")
print("\nWaiting for completion (this may take 5-10 minutes)...")

import time
while True:
    status = w.pipelines.get_update(pipeline_id=PIPELINE_ID, update_id=update.update_id)
    state = status.update.state.value if status.update.state else 'UNKNOWN'
    print(f"  State: {state}")
    if state in ('COMPLETED', 'FAILED', 'CANCELED'):
        break
    time.sleep(30)

print(f"\nPipeline refresh finished: {state}")

In [0]:
# Rebuild gold_anomaly_events_extended from the pipeline's gold_anomaly_events
# plus synthetic LANDING phase records to preserve the flight-phase analysis tab

gold_events = spark.read.table("genie_zeroops_mfg_catalog.default.gold_anomaly_events")
print(f"Pipeline gold_anomaly_events rows: {gold_events.count()}")

# The existing extended table adds LANDING phase records.
# For problem carriers, LANDING faults skew toward BRAKE_OVERHEAT
# For all carriers, LANDING events have higher vibration and shorter durations

import random
from pyspark.sql import Row
from datetime import datetime, timedelta

random.seed(99)

# Get flights that have anomalies for generating LANDING events
anom_flights = spark.read.table("genie_zeroops_mfg_catalog.default.gold_flight_summary").filter("has_anomaly = true")
anom_list = anom_flights.select('flight_id','airline','tail_number','aircraft_type','anomaly_type','anomaly_severity','departure_time').collect()

# Generate LANDING phase events for ~25% of anomaly flights
landing_rows = []
for row in anom_list:
    if random.random() < 0.30:  # 30% of anomaly flights get a landing event
        # Landing anomalies skew toward BRAKE_OVERHEAT
        if random.random() < 0.45:
            anom_type = 'BRAKE_OVERHEAT'
        elif random.random() < 0.3:
            anom_type = 'HYDRAULIC_LEAK'
        else:
            anom_type = row.anomaly_type  # keep original type
        
        # Higher severity for problem carriers at landing
        if row.airline in ('SkyWest Global', 'Atlas Pacific'):
            sev = random.choice(['CRITICAL','HIGH','HIGH','MEDIUM'])
        else:
            sev = random.choice(['MEDIUM','LOW','LOW','LOW'])
        
        event_start = row.departure_time + timedelta(hours=random.uniform(2, 5))
        landing_rows.append(Row(
            flight_id=row.flight_id,
            airline=row.airline,
            tail_number=row.tail_number,
            aircraft_type=row.aircraft_type,
            anomaly_type=anom_type,
            anomaly_severity=sev,
            flight_phase_at_peak='LANDING',
            peak_egt=float(random.uniform(750, 880)),
            peak_vibration=float(random.uniform(1.5, 3.8)),  # higher at landing
            min_oil_pressure_during_event=float(random.uniform(38, 55)),
            event_start_timestamp=event_start,
            event_duration_seconds=float(random.uniform(300, 3600)),  # shorter at landing
        ))

if landing_rows:
    landing_df = spark.createDataFrame(landing_rows)
else:
    landing_df = spark.createDataFrame([], gold_events.schema)

# Combine pipeline events (CLIMB/CRUISE) with synthetic LANDING events
combined = gold_events.unionByName(landing_df)

print(f"CLIMB/CRUISE events from pipeline: {gold_events.count()}")
print(f"Synthetic LANDING events: {landing_df.count()}")
print(f"Total extended events: {combined.count()}")

combined.write.mode('overwrite').saveAsTable('genie_zeroops_mfg_catalog.default.gold_anomaly_events_extended')
print("gold_anomaly_events_extended rebuilt.")

# Phase distribution
display(
    combined.groupBy('flight_phase_at_peak','anomaly_severity')
    .count()
    .orderBy('flight_phase_at_peak','anomaly_severity')
)

In [0]:
# Rebuild ml_maintenance_predictions from the updated gold_aircraft_health
import random
import builtins
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col

random.seed(77)

health = spark.read.table("genie_zeroops_mfg_catalog.default.gold_aircraft_health")
print(f"Aircraft in gold_aircraft_health: {health.count()}")

# For each aircraft, compute ML scores based on anomaly rate and carrier risk profile
CARRIER_RISK_BOOST = {
    'SkyWest Global': 0.25,
    'Atlas Pacific': 0.20,
    'Continental Wings': 0.10,
    'Horizon Express': 0.05,
    'Falcon Jet Airways': 0.03,
    'TransOcean Air': 0.0,
    'Meridian Airlines': 0.0,
    'Northern Star Airways': -0.05,
    'Aurora Airlines': -0.10,
    'Summit Aviation': -0.15,
}

rows = []
for r in health.collect():
    boost = CARRIER_RISK_BOOST.get(r.airline, 0.0)
    anom_rate = r.anomaly_rate if r.anomaly_rate else 0.0
    
    # ML anomaly probability: based on anomaly_rate + carrier risk + noise
    base_prob = builtins.min(1.0, builtins.max(0.0, anom_rate * 3.0 + boost + random.uniform(-0.05, 0.05)))
    
    # Health risk score from gold_aircraft_health
    hrs = r.health_risk_score if r.health_risk_score else 8.0
    
    # Maintenance urgency factor
    days_until = r.days_until_next_maintenance if r.days_until_next_maintenance else 180
    maint_urgency = builtins.max(0, (180 - days_until) / 180.0) * 20  # 0-20 points
    
    # Composite priority score (0-100)
    priority = builtins.min(100, base_prob * 40 + hrs * 0.5 + maint_urgency + random.uniform(0, 5))
    
    # Risk category
    if priority > 75:
        risk_cat = 'CRITICAL'
    elif priority > 50:
        risk_cat = 'HIGH'
    elif priority > 25:
        risk_cat = 'MEDIUM'
    else:
        risk_cat = 'LOW'
    
    rows.append(Row(
        tail_number=r.tail_number,
        airline=r.airline,
        aircraft_type=r.aircraft_type,
        ml_anomaly_probability=builtins.round(base_prob, 3),
        health_risk_score=builtins.round(hrs, 1),
        anomaly_rate=builtins.round(anom_rate, 3),
        days_since_last_maintenance=int(r.days_since_last_maintenance) if r.days_since_last_maintenance else 0,
        days_until_next_maintenance=int(r.days_until_next_maintenance) if r.days_until_next_maintenance else 0,
        total_flights=int(r.total_flights) if r.total_flights else 0,
        anomaly_flight_count=int(r.anomaly_flight_count) if r.anomaly_flight_count else 0,
        maintenance_priority_score=builtins.round(priority, 1),
        risk_category=risk_cat,
    ))

ml_df = spark.createDataFrame(rows)
# Keep the original column types (INT for day counts) so the Lakebase synced copy stays compatible
ml_df = ml_df.withColumn('days_since_last_maintenance', F.col('days_since_last_maintenance').cast('int')) \
             .withColumn('days_until_next_maintenance', F.col('days_until_next_maintenance').cast('int'))
ml_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('genie_zeroops_mfg_catalog.default.ml_maintenance_predictions')
print(f"ml_maintenance_predictions rebuilt: {ml_df.count()} aircraft")

# Risk distribution
display(
    ml_df.groupBy('risk_category').count().orderBy(
        when(col('risk_category')=='CRITICAL',1)
        .when(col('risk_category')=='HIGH',2)
        .when(col('risk_category')=='MEDIUM',3)
        .otherwise(4)
    )
)

In [0]:
# Snapshot-mode Lakebase synced tables only refresh when their sync pipelines run.
# Trigger each one so the dashboard (*_synced tables) picks up the reprocessed data.
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
SYNCED = {
    "gold_flight_summary_synced": "f26d1654-890e-48f8-8476-6e41f5d36461",
    "gold_airline_operations_synced": "f37b9ef5-b99c-4a50-a6de-136bd6dca13b",
    "gold_aircraft_health_synced": "8ab6c50a-7f55-42d9-a782-d9f4f1552d3c",
    "gold_anomaly_events_synced": "c5f88868-5351-444b-90b5-746c3207bc50",
    "ml_maintenance_predictions_synced": "48245b5a-ace4-44bb-99c1-d15f0682fa48",
}

updates = {name: w.pipelines.start_update(pipeline_id=pid).update_id for name, pid in SYNCED.items()}
print("Sync updates started:", updates)

pending = dict(updates)
results = {}
while pending:
    time.sleep(20)
    for name, uid in list(pending.items()):
        st = w.pipelines.get_update(pipeline_id=SYNCED[name], update_id=uid).update.state.value
        if st in ("COMPLETED", "FAILED", "CANCELED"):
            results[name] = st
            pending.pop(name)
for name, st in results.items():
    print(f"{name}: {st}")

In [0]:
%sql
-- Airline-level summary from gold_flight_summary
SELECT airline, 
  COUNT(*) flights,
  SUM(CASE WHEN has_anomaly THEN 1 ELSE 0 END) anomalies,
  ROUND(AVG(CASE WHEN has_anomaly THEN 1.0 ELSE 0.0 END)*100, 1) anomaly_pct,
  ROUND(SUM(flight_duration_hours), 0) total_hrs
FROM genie_zeroops_mfg_catalog.default.gold_flight_summary
GROUP BY airline ORDER BY anomaly_pct DESC

In [0]:
%sql
-- Monthly anomaly trend for SkyWest and Atlas (should be rising Jul-Oct 2026)
SELECT airline,
  DATE_TRUNC('month', departure_time) AS month,
  COUNT(*) flights,
  SUM(CASE WHEN has_anomaly THEN 1 ELSE 0 END) anomalies,
  ROUND(AVG(CASE WHEN has_anomaly THEN 1.0 ELSE 0.0 END)*100, 1) anomaly_pct
FROM genie_zeroops_mfg_catalog.default.gold_flight_summary
WHERE airline IN ('SkyWest Global', 'Atlas Pacific')
GROUP BY airline, DATE_TRUNC('month', departure_time)
ORDER BY airline, month

In [0]:
%sql
-- Flight phase distribution in anomaly events
SELECT flight_phase_at_peak, anomaly_severity, COUNT(*) cnt
FROM genie_zeroops_mfg_catalog.default.gold_anomaly_events_extended
GROUP BY flight_phase_at_peak, anomaly_severity
ORDER BY flight_phase_at_peak, anomaly_severity